In [1]:
# =============================================================================================
# AI651 Assignment 1 - Task 2: Autoformer leaderboard (ONE CELL)
# Trains 5 candidate Autoformer designs over several seeds on a chronological validation split,
# picks the best (accuracy first, then size), runs the external-data ablation, retrains the
# winner on the full history and writes the 168 forecasts ready to paste.
# Decomposition + Auto-Correlation are reused from my Question 1 implementation.
# =============================================================================================
import json, math, time, os
from dataclasses import dataclass, replace, asdict
from pathlib import Path
import numpy as np, pandas as pd, torch
from torch import nn
from torch.nn import functional as F

# ---------------- settings you may change ----------------
SEEDS = [0, 1]            # seeds per candidate (use [0, 1, 2] if you have time)
RUN_ABLATION = True       # with / without the external file, needed for the report
QUICK_TEST = False        # True = tiny run just to check the cell works (~1 min)
FINAL_MODE = "refit"      # "refit" = retrain winner on full history; "reuse" = use the validation model
torch.set_num_threads(os.cpu_count() or 4)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Find the data folder (works from Question 1 or Question 2 folder)
for d in [Path("."), Path("Data"), Path("../Question 2 - Leaderboard/Data"), Path("Question 2 - Leaderboard/Data")]:
    if (d / "student_train.csv").exists():
        DATA = d; break
else:
    raise FileNotFoundError("Put this cell next to student_train.csv or set DATA manually.")
OUT = Path("task2_results"); OUT.mkdir(exist_ok=True)
H_DATA, H_TOTAL = 43656, 43824

@dataclass
class Config:
    name: str = "base"
    seq_len: int = 168; label_len: int = 48; pred_len: int = 168
    d_model: int = 32; n_heads: int = 4; d_ff: int = 64
    e_layers: int = 1; d_layers: int = 1
    moving_avg: int = 25; factor: int = 1; dropout: float = 0.1
    covariates: str = "past+future"      # "none" | "past" | "past+future"
    target_transform: str = "log1p"      # "log1p" | "none"
    batch_size: int = 64; lr: float = 1e-3
    max_epochs: int = 8; patience: int = 2
    train_stride: int = 4; val_weeks: int = 10; val_stride: int = 24

# ---------------- the 5 candidate designs ----------------
BASE = Config()
CANDIDATES = [
    replace(BASE, name="A_base_d32"),                                         # 1 enc + 1 dec, width 32
    replace(BASE, name="B_tiny_d16", d_model=16, d_ff=32),                    # half width, ~4x fewer params
    replace(BASE, name="C_long_input_336", seq_len=336, label_len=96),        # two weeks of history
    replace(BASE, name="D_deeper_2enc", e_layers=2),                          # extra encoder layer
    replace(BASE, name="E_raw_target", target_transform="none"),             # no log transform
]
if QUICK_TEST:
    CANDIDATES = [replace(c, max_epochs=1, train_stride=64) for c in CANDIDATES[:2]]
    SEEDS = [0]

# ---------------- data ----------------
CONT = ["feature_A","feature_B","feature_C","feature_D","feature_E","feature_F"]
BIN = ["feature_G","feature_H","feature_I","feature_J"]
y_all = pd.read_csv(DATA / "student_train.csv")["value"].to_numpy(np.float64)
ext = pd.read_csv(DATA / "optional_external_data.csv")
assert len(y_all) == H_DATA and len(ext) == H_TOTAL

def calendar(n):                       # daily (24) and weekly (168) phase, found by autocorrelation
    t = np.arange(1, n + 1)
    return np.stack([f(2*np.pi*t/p) for p in (24, 168) for f in (np.sin, np.cos)], 1)

def covariates(fit_end):
    X = ext[CONT + BIN].astype(np.float64).copy()
    for c in ["feature_D","feature_E","feature_F"]:       # long-tailed -> log1p
        X[c] = np.log1p(X[c].clip(lower=0))
    mu, sd = X[CONT].iloc[:fit_end].mean(), X[CONT].iloc[:fit_end].std() + 1e-8
    X[CONT] = (X[CONT] - mu) / sd                         # train-only statistics
    return X.to_numpy(np.float32)

class Scaler:
    def __init__(s, y, tf):
        s.tf = tf; z = s.f(y); s.mu, s.sd = float(z.mean()), float(z.std() + 1e-8)
    def f(s, y): return np.log1p(np.clip(y, 0, None)) if s.tf == "log1p" else y
    def enc(s, y): return ((s.f(y) - s.mu) / s.sd).astype(np.float32)
    def dec(s, z):
        v = z * s.sd + s.mu
        return np.clip(np.expm1(v) if s.tf == "log1p" else v, 0, None)

def prepare(c, fit_end):
    sc = Scaler(y_all[:fit_end], c.target_transform)
    z = np.zeros(H_TOTAL, np.float32); z[:H_DATA] = sc.enc(y_all)
    cal = calendar(H_TOTAL).astype(np.float32)
    full = np.concatenate([covariates(fit_end), cal], 1)
    em, dm = {"none": (cal, cal), "past": (full, cal), "past+future": (full, full)}[c.covariates]
    return sc, z, em, dm

class Windows(torch.utils.data.Dataset):
    def __init__(s, z, em, dm, origins, c): s.z, s.em, s.dm, s.o, s.c = z, em, dm, origins, c
    def __len__(s): return len(s.o)
    def __getitem__(s, i):
        o, c = int(s.o[i]), s.c
        y = s.z[o:o+c.pred_len] if o + c.pred_len <= H_DATA else np.zeros(c.pred_len, np.float32)
        return (torch.from_numpy(s.z[o-c.seq_len:o, None].copy()), torch.from_numpy(s.em[o-c.seq_len:o]),
                torch.from_numpy(s.dm[o-c.label_len:o+c.pred_len]), torch.from_numpy(np.ascontiguousarray(y)))

# ---------------- Autoformer (decomposition + auto-correlation from Question 1) ----------------
class SeriesDecomposition(nn.Module):           # Q1 Part 2
    def __init__(s, k): super().__init__(); s.k = k
    def forward(s, x):
        r = s.k // 2
        p = torch.cat([x[:, :1].repeat(1, r, 1), x, x[:, -1:].repeat(1, r, 1)], 1)
        trend = F.avg_pool1d(p.transpose(1, 2), s.k, stride=1).transpose(1, 2)
        return x - trend, trend

def delay_scores(q, k):                          # Q1 Part 3 (supplied FFT correlation)
    q = q - q.mean(-1, keepdim=True); k = k - k.mean(-1, keepdim=True)
    return torch.fft.irfft(torch.fft.rfft(q, dim=-1) * torch.fft.rfft(k, dim=-1).conj(),
                           n=q.shape[-1], dim=-1).mean((1, 2))

def aggregate_delays(v, delays, w):              # Q1 Part 3 (my implementation)
    B, Hh, Fh, L = v.shape; K = delays.shape[1]
    idx = (torch.arange(L, device=v.device)[None, None] - delays[:, :, None]) % L
    idx = idx[:, :, None, None, :].expand(B, K, Hh, Fh, L)
    return (w[:, :, None, None, None] * torch.gather(v[:, None].expand(B, K, Hh, Fh, L), -1, idx)).sum(1)

class AutoCorrelation(nn.Module):
    def __init__(s, d, h, factor):
        super().__init__(); s.h, s.factor = h, factor
        s.q, s.k, s.v, s.o = (nn.Linear(d, d) for _ in range(4))
    def forward(s, qx, kx, vx):
        B, L, d = qx.shape; S = kx.shape[1]
        if L > S:
            pad = qx.new_zeros(B, L - S, d); kx, vx = torch.cat([kx, pad], 1), torch.cat([vx, pad], 1)
        else:
            kx, vx = kx[:, :L], vx[:, :L]
        q, k, v = (p(x).view(B, L, s.h, d // s.h).permute(0, 2, 3, 1) for p, x in ((s.q, qx), (s.k, kx), (s.v, vx)))
        sel, delays = delay_scores(q, k).topk(max(1, int(s.factor * math.log(L))), -1)
        return s.o(aggregate_delays(v, delays, sel.softmax(-1)).permute(0, 3, 1, 2).reshape(B, L, d))

class Embedding(nn.Module):                      # value conv + known-feature linear, no positions
    def __init__(s, cm, d, p):
        super().__init__()
        s.val = nn.Conv1d(1, d, 3, padding=1, padding_mode="circular", bias=False)
        s.mark = nn.Linear(cm, d, bias=False); s.drop = nn.Dropout(p)
    def forward(s, x, m): return s.drop(s.val(x.transpose(1, 2)).transpose(1, 2) + s.mark(m))

def ffn(c): return nn.Sequential(nn.Linear(c.d_model, c.d_ff, bias=False), nn.GELU(),
                                 nn.Dropout(c.dropout), nn.Linear(c.d_ff, c.d_model, bias=False))

class EncoderLayer(nn.Module):
    def __init__(s, c):
        super().__init__(); s.ac = AutoCorrelation(c.d_model, c.n_heads, c.factor); s.ff = ffn(c)
        s.d1, s.d2 = SeriesDecomposition(c.moving_avg), SeriesDecomposition(c.moving_avg); s.drop = nn.Dropout(c.dropout)
    def forward(s, x):
        x, _ = s.d1(x + s.drop(s.ac(x, x, x)))
        x, _ = s.d2(x + s.drop(s.ff(x)))
        return x

class SeasonalNorm(nn.Module):
    def __init__(s, d): super().__init__(); s.ln = nn.LayerNorm(d)
    def forward(s, x): x = s.ln(x); return x - x.mean(1, keepdim=True)

class DecoderLayer(nn.Module):
    def __init__(s, c):
        super().__init__()
        s.sa, s.ca = AutoCorrelation(c.d_model, c.n_heads, c.factor), AutoCorrelation(c.d_model, c.n_heads, c.factor)
        s.ff = ffn(c); s.dec = nn.ModuleList(SeriesDecomposition(c.moving_avg) for _ in range(3))
        s.trend = nn.Conv1d(c.d_model, 1, 3, padding=1, padding_mode="circular", bias=False); s.drop = nn.Dropout(c.dropout)
    def forward(s, x, enc):
        x, t1 = s.dec[0](x + s.drop(s.sa(x, x, x)))
        x, t2 = s.dec[1](x + s.drop(s.ca(x, enc, enc)))
        x, t3 = s.dec[2](x + s.drop(s.ff(x)))
        return x, s.trend((t1 + t2 + t3).transpose(1, 2)).transpose(1, 2)

class Autoformer(nn.Module):
    def __init__(s, c, ce, cd):
        super().__init__(); s.c = c; s.decomp = SeriesDecomposition(c.moving_avg)
        s.ee, s.de = Embedding(ce, c.d_model, c.dropout), Embedding(cd, c.d_model, c.dropout)
        s.enc = nn.ModuleList(EncoderLayer(c) for _ in range(c.e_layers)); s.en = SeasonalNorm(c.d_model)
        s.dec = nn.ModuleList(DecoderLayer(c) for _ in range(c.d_layers)); s.dn = SeasonalNorm(c.d_model)
        s.proj = nn.Linear(c.d_model, 1)
    def forward(s, x, me, md):
        c = s.c; seas, trend = s.decomp(x)
        mean = x.mean(1, keepdim=True).repeat(1, c.pred_len, 1)
        trend_acc = torch.cat([trend[:, -c.label_len:], mean], 1)                 # trend init
        seas_init = torch.cat([seas[:, -c.label_len:], torch.zeros_like(mean)], 1)  # seasonal init
        e = s.ee(x, me)
        for l in s.enc: e = l(e)
        e = s.en(e); d = s.de(seas_init, md)
        for l in s.dec:
            d, t = l(d, e); trend_acc = trend_acc + t                             # progressive trend
        return (trend_acc + s.proj(s.dn(d)))[:, -c.pred_len:, 0]

n_params = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)

# ---------------- training / evaluation ----------------
def metrics(p, t):
    e = p - t
    return {"RMSE": float(np.sqrt((e**2).mean())), "MAE": float(np.abs(e).mean()),
            "sMAPE": float(100 * np.mean(2*np.abs(e) / (np.abs(t) + np.abs(p) + 1e-8)))}

@torch.no_grad()
def predict(m, ds):
    m.eval()
    return np.concatenate([m(x.to(DEVICE), a.to(DEVICE), b.to(DEVICE)).cpu().numpy()
                           for x, a, b, _ in torch.utils.data.DataLoader(ds, batch_size=256)])

def val_origins(c): return np.arange(H_DATA - c.val_weeks*168, H_DATA - c.pred_len + 1, c.val_stride)

def train(c, seed, fit_end, vo=None, epochs=None, tag=""):
    torch.manual_seed(seed); np.random.seed(seed)
    sc, z, em, dm = prepare(c, fit_end)
    tr = Windows(z, em, dm, np.arange(c.seq_len, fit_end - c.pred_len + 1, c.train_stride), c)
    dl = torch.utils.data.DataLoader(tr, batch_size=c.batch_size, shuffle=True, generator=torch.Generator().manual_seed(seed))
    m = Autoformer(c, em.shape[1], dm.shape[1]).to(DEVICE); opt = torch.optim.Adam(m.parameters(), lr=c.lr)
    va = Windows(z, em, dm, vo, c) if vo is not None else None
    tv = np.stack([y_all[o:o+c.pred_len] for o in vo]) if vo is not None else None
    best, state, bad, log = 1e18, None, 0, []
    for ep in range(1, (epochs or c.max_epochs) + 1):
        m.train(); t0 = time.time()
        for x, a, b, t in dl:
            opt.zero_grad(); F.mse_loss(m(x.to(DEVICE), a.to(DEVICE), b.to(DEVICE)), t.to(DEVICE)).backward(); opt.step()
        for g in opt.param_groups: g["lr"] *= 0.5
        row = {"epoch": ep, "sec": round(time.time() - t0, 1)}
        if va is not None:
            row.update(metrics(sc.dec(predict(m, va)), tv))
            if row["RMSE"] < best: best, bad, state = row["RMSE"], 0, {k: v.clone() for k, v in m.state_dict().items()}
            else: bad += 1
        log.append(row); print(f"  {tag} epoch {ep}: " + ", ".join(f"{k}={v:.2f}" if isinstance(v, float) else f"{k}={v}" for k, v in row.items() if k != "epoch"), flush=True)
        if va is not None and bad >= c.patience: break
    if state: m.load_state_dict(state)
    return m, sc, log, (z, em, dm)

def evaluate(c, seed):
    vo = val_origins(c)
    m, sc, log, _ = train(c, seed, int(vo[0]), vo, tag=f"{c.name}/seed{seed}")
    b = min(log, key=lambda r: r["RMSE"])
    return {"model": c.name, "covariates": c.covariates, "seed": seed, "RMSE": b["RMSE"], "MAE": b["MAE"],
            "sMAPE": b["sMAPE"], "best_epoch": b["epoch"], "epochs_run": len(log), "params": n_params(m)}

# ---------------- 1) reference baselines on the same validation blocks ----------------
vo0 = val_origins(BASE); T = np.stack([y_all[o:o+168] for o in vo0])
base_rows = [{"model": "baseline: training mean", **metrics(np.full_like(T, y_all[:vo0[0]].mean()), T)},
             {"model": "baseline: persistence", **metrics(np.stack([np.full(168, y_all[o-1]) for o in vo0]), T)},
             {"model": "baseline: last-week mean", **metrics(np.stack([np.full(168, y_all[o-168:o].mean()) for o in vo0]), T)}]
print(pd.DataFrame(base_rows).round(2).to_string(index=False))

# ---------------- 2) search the 5 candidates ----------------
rows = []
for c in CANDIDATES:
    for s in SEEDS:
        rows.append(evaluate(c, s)); print(rows[-1], flush=True)
runs = pd.DataFrame(rows); runs.to_csv(OUT / "candidate_runs.csv", index=False)
summary = (runs.groupby("model").agg(RMSE_mean=("RMSE", "mean"), RMSE_std=("RMSE", "std"), MAE=("MAE", "mean"),
           sMAPE=("sMAPE", "mean"), params=("params", "first"), best_epoch=("best_epoch", "median"))
           .sort_values("RMSE_mean").round(2))
summary.to_csv(OUT / "candidate_summary.csv"); print("\nCANDIDATES (validation, mean over seeds)\n", summary.to_string())

# Selection: accuracy first; among designs within 1% of the best mean RMSE, take the fewest parameters
tol = summary["RMSE_mean"].min() * 1.01
winner_name = summary[summary["RMSE_mean"] <= tol].sort_values("params").index[0]
WIN = next(c for c in CANDIDATES if c.name == winner_name)
E_fit = max(1, int(round(summary.loc[winner_name, "best_epoch"])))
print(f"\nWINNER: {winner_name}  (params={summary.loc[winner_name,'params']}, epochs for final fit={E_fit})")

# ---------------- 3) ablation of the external file on the winner ----------------
if RUN_ABLATION:
    ab = []
    for cov in ["none", "past", "past+future"]:
        c = replace(WIN, covariates=cov, name=f"{WIN.name}[{cov}]")
        for s in SEEDS:
            r = evaluate(c, s) if not (cov == WIN.covariates) else runs[(runs.model == WIN.name) & (runs.seed == s)].iloc[0].to_dict()
            r["covariates"] = cov; ab.append(r)
    ab = pd.DataFrame(ab); ab.to_csv(OUT / "ablation_runs.csv", index=False)
    ab_sum = ab.groupby("covariates")[["RMSE", "MAE", "sMAPE"]].agg(["mean", "std"]).round(2)
    ab_sum.to_csv(OUT / "ablation_summary.csv"); print("\nABLATION (external file)\n", ab_sum.to_string())

# ---------------- 4) final model and the 168 forecasts ----------------
seed = SEEDS[0]
if FINAL_MODE == "refit":
    sel_epochs = int(runs[(runs.model == WIN.name) & (runs.seed == seed)].epochs_run.iloc[0])
    m, sc, log, (z, em, dm) = train(WIN, seed, H_DATA, None, epochs=E_fit, tag="FINAL")
    E_declared = sel_epochs + len(log)     # handout: early-stopping run + refit both count
else:
    vo = val_origins(WIN); m, sc, log, (z, em, dm) = train(WIN, seed, int(vo[0]), vo, tag="FINAL")
    E_declared = len(log)
pred = sc.dec(predict(m, Windows(z, em, dm, np.array([H_DATA]), WIN)))[0]
assert pred.shape == (168,) and np.isfinite(pred).all()
P_declared = n_params(m)
pd.DataFrame({"time_idx": np.arange(H_DATA + 1, H_TOTAL + 1), "value": pred}).to_csv(OUT / "submission.csv", index=False)
(OUT / "leaderboard_paste.txt").write_text(", ".join(f"{v:.3f}" for v in pred))
json.dump({"winner": WIN.name, "config": asdict(WIN), "seed": seed, "P_params": P_declared,
           "E_epochs": E_declared, "final_mode": FINAL_MODE}, open(OUT / "submission_info.json", "w"), indent=2)
print(f"\nDONE. Declare P = {P_declared} parameters, E = {E_declared} epochs.")
print(f"Paste file: {OUT/'leaderboard_paste.txt'}  (168 values, time_idx 43657 -> 43824)")
print("First 5 forecasts:", np.round(pred[:5], 1), "...  last:", round(float(pred[-1]), 1))

                   model   RMSE    MAE  sMAPE
 baseline: training mean 109.42  86.64  92.89
   baseline: persistence 157.13 116.79 101.68
baseline: last-week mean 117.31  92.25  93.37
  A_base_d32/seed0 epoch 1: sec=16.70, RMSE=99.18, MAE=61.36, sMAPE=59.65
  A_base_d32/seed0 epoch 2: sec=16.60, RMSE=86.70, MAE=51.93, sMAPE=53.51
  A_base_d32/seed0 epoch 3: sec=16.80, RMSE=85.03, MAE=50.60, sMAPE=52.42
  A_base_d32/seed0 epoch 4: sec=16.70, RMSE=84.80, MAE=50.48, sMAPE=52.38
  A_base_d32/seed0 epoch 5: sec=16.70, RMSE=85.95, MAE=50.78, sMAPE=52.84
  A_base_d32/seed0 epoch 6: sec=16.80, RMSE=87.28, MAE=51.34, sMAPE=53.22
{'model': 'A_base_d32', 'covariates': 'past+future', 'seed': 0, 'RMSE': 84.79505084698175, 'MAE': 50.476357729728555, 'sMAPE': 52.38189179631373, 'best_epoch': 4, 'epochs_run': 6, 'params': 22209}
  A_base_d32/seed1 epoch 1: sec=17.00, RMSE=96.77, MAE=62.46, sMAPE=62.64
  A_base_d32/seed1 epoch 2: sec=17.20, RMSE=86.22, MAE=56.57, sMAPE=57.37
  A_base_d32/seed1 epoch 3:

## Round 2 

In [2]:
# =============================================================================================
# AI651 Assignment 1 - Task 2: Autoformer leaderboard (ONE CELL)
# Trains 5 candidate Autoformer designs over several seeds on a chronological validation split,
# picks the best (accuracy first, then size), runs the external-data ablation, retrains the
# winner on the full history and writes the 168 forecasts ready to paste.
# Decomposition + Auto-Correlation are reused from my Question 1 implementation.
# =============================================================================================
import json, math, time, os
from dataclasses import dataclass, replace, asdict
from pathlib import Path
import numpy as np, pandas as pd, torch
from torch import nn
from torch.nn import functional as F

# ---------------- settings you may change ----------------
SEEDS = [0, 1, 2]         # 3 seeds: 2 were not enough to separate designs (see notes)
RUN_ABLATION = True       # with / without the external file, needed for the report
QUICK_TEST = False        # True = tiny run just to check the cell works (~1 min)
FINAL_MODE = "reuse"      # "reuse" = validation-checked model of the best seed (safe); "refit" = blind retrain on full history
torch.set_num_threads(os.cpu_count() or 4)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Find the data folder (works from Question 1 or Question 2 folder)
for d in [Path("."), Path("Data"), Path("../Question 2 - Leaderboard/Data"), Path("Question 2 - Leaderboard/Data")]:
    if (d / "student_train.csv").exists():
        DATA = d; break
else:
    raise FileNotFoundError("Put this cell next to student_train.csv or set DATA manually.")
OUT = Path("task2_results"); OUT.mkdir(exist_ok=True)
H_DATA, H_TOTAL = 43656, 43824

@dataclass
class Config:
    name: str = "base"
    seq_len: int = 168; label_len: int = 48; pred_len: int = 168
    d_model: int = 32; n_heads: int = 4; d_ff: int = 64
    e_layers: int = 1; d_layers: int = 1
    moving_avg: int = 25; factor: int = 1; dropout: float = 0.1
    covariates: str = "past+future"      # "none" | "past" | "past+future"
    target_transform: str = "log1p"      # "log1p" | "none"
    batch_size: int = 64; lr: float = 1e-3
    max_epochs: int = 8; patience: int = 2
    grad_clip: float = 1.0               # FIX: stops spikes in the raw target from blowing up training
    train_stride: int = 4; val_weeks: int = 10; val_stride: int = 24

# ---------------- the 5 candidate designs ----------------
BASE = Config()
# Round 1 showed: raw target best on RMSE but unstable; 336 input stable and good; tiny and deeper worse.
CANDIDATES = [
    replace(BASE, name="A_log_d32"),                                                       # round-1 reference
    replace(BASE, name="C_log_336", seq_len=336, label_len=96),                            # round-1 runner-up
    replace(BASE, name="E_raw_d32", target_transform="none"),                              # round-1 winner, now clipped
    replace(BASE, name="F_raw_d32_lr5e-4", target_transform="none", lr=5e-4),              # slower, steadier
    replace(BASE, name="G_raw_336_lr5e-4", target_transform="none", lr=5e-4, seq_len=336, label_len=96),
]
if QUICK_TEST:
    CANDIDATES = [replace(c, max_epochs=1, train_stride=64) for c in CANDIDATES[:2]]
    SEEDS = [0]

# ---------------- data ----------------
CONT = ["feature_A","feature_B","feature_C","feature_D","feature_E","feature_F"]
BIN = ["feature_G","feature_H","feature_I","feature_J"]
y_all = pd.read_csv(DATA / "student_train.csv")["value"].to_numpy(np.float64)
ext = pd.read_csv(DATA / "optional_external_data.csv")
assert len(y_all) == H_DATA and len(ext) == H_TOTAL

def calendar(n):                       # daily (24) and weekly (168) phase, found by autocorrelation
    t = np.arange(1, n + 1)
    return np.stack([f(2*np.pi*t/p) for p in (24, 168) for f in (np.sin, np.cos)], 1)

def covariates(fit_end):
    X = ext[CONT + BIN].astype(np.float64).copy()
    for c in ["feature_D","feature_E","feature_F"]:       # long-tailed -> log1p
        X[c] = np.log1p(X[c].clip(lower=0))
    mu, sd = X[CONT].iloc[:fit_end].mean(), X[CONT].iloc[:fit_end].std() + 1e-8
    X[CONT] = (X[CONT] - mu) / sd                         # train-only statistics
    return X.to_numpy(np.float32)

class Scaler:
    def __init__(s, y, tf):
        s.tf = tf; z = s.f(y); s.mu, s.sd = float(z.mean()), float(z.std() + 1e-8)
    def f(s, y): return np.log1p(np.clip(y, 0, None)) if s.tf == "log1p" else y
    def enc(s, y): return ((s.f(y) - s.mu) / s.sd).astype(np.float32)
    def dec(s, z):
        v = z * s.sd + s.mu
        return np.clip(np.expm1(v) if s.tf == "log1p" else v, 0, None)

def prepare(c, fit_end):
    sc = Scaler(y_all[:fit_end], c.target_transform)
    z = np.zeros(H_TOTAL, np.float32); z[:H_DATA] = sc.enc(y_all)
    cal = calendar(H_TOTAL).astype(np.float32)
    full = np.concatenate([covariates(fit_end), cal], 1)
    em, dm = {"none": (cal, cal), "past": (full, cal), "past+future": (full, full)}[c.covariates]
    return sc, z, em, dm

class Windows(torch.utils.data.Dataset):
    def __init__(s, z, em, dm, origins, c): s.z, s.em, s.dm, s.o, s.c = z, em, dm, origins, c
    def __len__(s): return len(s.o)
    def __getitem__(s, i):
        o, c = int(s.o[i]), s.c
        y = s.z[o:o+c.pred_len] if o + c.pred_len <= H_DATA else np.zeros(c.pred_len, np.float32)
        return (torch.from_numpy(s.z[o-c.seq_len:o, None].copy()), torch.from_numpy(s.em[o-c.seq_len:o]),
                torch.from_numpy(s.dm[o-c.label_len:o+c.pred_len]), torch.from_numpy(np.ascontiguousarray(y)))

# ---------------- Autoformer (decomposition + auto-correlation from Question 1) ----------------
class SeriesDecomposition(nn.Module):           # Q1 Part 2
    def __init__(s, k): super().__init__(); s.k = k
    def forward(s, x):
        r = s.k // 2
        p = torch.cat([x[:, :1].repeat(1, r, 1), x, x[:, -1:].repeat(1, r, 1)], 1)
        trend = F.avg_pool1d(p.transpose(1, 2), s.k, stride=1).transpose(1, 2)
        return x - trend, trend

def delay_scores(q, k):                          # Q1 Part 3 (supplied FFT correlation)
    q = q - q.mean(-1, keepdim=True); k = k - k.mean(-1, keepdim=True)
    return torch.fft.irfft(torch.fft.rfft(q, dim=-1) * torch.fft.rfft(k, dim=-1).conj(),
                           n=q.shape[-1], dim=-1).mean((1, 2))

def aggregate_delays(v, delays, w):              # Q1 Part 3 (my implementation)
    B, Hh, Fh, L = v.shape; K = delays.shape[1]
    idx = (torch.arange(L, device=v.device)[None, None] - delays[:, :, None]) % L
    idx = idx[:, :, None, None, :].expand(B, K, Hh, Fh, L)
    return (w[:, :, None, None, None] * torch.gather(v[:, None].expand(B, K, Hh, Fh, L), -1, idx)).sum(1)

class AutoCorrelation(nn.Module):
    def __init__(s, d, h, factor):
        super().__init__(); s.h, s.factor = h, factor
        s.q, s.k, s.v, s.o = (nn.Linear(d, d) for _ in range(4))
    def forward(s, qx, kx, vx):
        B, L, d = qx.shape; S = kx.shape[1]
        if L > S:
            pad = qx.new_zeros(B, L - S, d); kx, vx = torch.cat([kx, pad], 1), torch.cat([vx, pad], 1)
        else:
            kx, vx = kx[:, :L], vx[:, :L]
        q, k, v = (p(x).view(B, L, s.h, d // s.h).permute(0, 2, 3, 1) for p, x in ((s.q, qx), (s.k, kx), (s.v, vx)))
        sel, delays = delay_scores(q, k).topk(max(1, int(s.factor * math.log(L))), -1)
        return s.o(aggregate_delays(v, delays, sel.softmax(-1)).permute(0, 3, 1, 2).reshape(B, L, d))

class Embedding(nn.Module):                      # value conv + known-feature linear, no positions
    def __init__(s, cm, d, p):
        super().__init__()
        s.val = nn.Conv1d(1, d, 3, padding=1, padding_mode="circular", bias=False)
        s.mark = nn.Linear(cm, d, bias=False); s.drop = nn.Dropout(p)
    def forward(s, x, m): return s.drop(s.val(x.transpose(1, 2)).transpose(1, 2) + s.mark(m))

def ffn(c): return nn.Sequential(nn.Linear(c.d_model, c.d_ff, bias=False), nn.GELU(),
                                 nn.Dropout(c.dropout), nn.Linear(c.d_ff, c.d_model, bias=False))

class EncoderLayer(nn.Module):
    def __init__(s, c):
        super().__init__(); s.ac = AutoCorrelation(c.d_model, c.n_heads, c.factor); s.ff = ffn(c)
        s.d1, s.d2 = SeriesDecomposition(c.moving_avg), SeriesDecomposition(c.moving_avg); s.drop = nn.Dropout(c.dropout)
    def forward(s, x):
        x, _ = s.d1(x + s.drop(s.ac(x, x, x)))
        x, _ = s.d2(x + s.drop(s.ff(x)))
        return x

class SeasonalNorm(nn.Module):
    def __init__(s, d): super().__init__(); s.ln = nn.LayerNorm(d)
    def forward(s, x): x = s.ln(x); return x - x.mean(1, keepdim=True)

class DecoderLayer(nn.Module):
    def __init__(s, c):
        super().__init__()
        s.sa, s.ca = AutoCorrelation(c.d_model, c.n_heads, c.factor), AutoCorrelation(c.d_model, c.n_heads, c.factor)
        s.ff = ffn(c); s.dec = nn.ModuleList(SeriesDecomposition(c.moving_avg) for _ in range(3))
        s.trend = nn.Conv1d(c.d_model, 1, 3, padding=1, padding_mode="circular", bias=False); s.drop = nn.Dropout(c.dropout)
    def forward(s, x, enc):
        x, t1 = s.dec[0](x + s.drop(s.sa(x, x, x)))
        x, t2 = s.dec[1](x + s.drop(s.ca(x, enc, enc)))
        x, t3 = s.dec[2](x + s.drop(s.ff(x)))
        return x, s.trend((t1 + t2 + t3).transpose(1, 2)).transpose(1, 2)

class Autoformer(nn.Module):
    def __init__(s, c, ce, cd):
        super().__init__(); s.c = c; s.decomp = SeriesDecomposition(c.moving_avg)
        s.ee, s.de = Embedding(ce, c.d_model, c.dropout), Embedding(cd, c.d_model, c.dropout)
        s.enc = nn.ModuleList(EncoderLayer(c) for _ in range(c.e_layers)); s.en = SeasonalNorm(c.d_model)
        s.dec = nn.ModuleList(DecoderLayer(c) for _ in range(c.d_layers)); s.dn = SeasonalNorm(c.d_model)
        s.proj = nn.Linear(c.d_model, 1)
    def forward(s, x, me, md):
        c = s.c; seas, trend = s.decomp(x)
        mean = x.mean(1, keepdim=True).repeat(1, c.pred_len, 1)
        trend_acc = torch.cat([trend[:, -c.label_len:], mean], 1)                 # trend init
        seas_init = torch.cat([seas[:, -c.label_len:], torch.zeros_like(mean)], 1)  # seasonal init
        e = s.ee(x, me)
        for l in s.enc: e = l(e)
        e = s.en(e); d = s.de(seas_init, md)
        for l in s.dec:
            d, t = l(d, e); trend_acc = trend_acc + t                             # progressive trend
        return (trend_acc + s.proj(s.dn(d)))[:, -c.pred_len:, 0]

n_params = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)

# ---------------- training / evaluation ----------------
def metrics(p, t):
    e = p - t
    return {"RMSE": float(np.sqrt((e**2).mean())), "MAE": float(np.abs(e).mean()),
            "sMAPE": float(100 * np.mean(2*np.abs(e) / (np.abs(t) + np.abs(p) + 1e-8)))}

@torch.no_grad()
def predict(m, ds):
    m.eval()
    return np.concatenate([m(x.to(DEVICE), a.to(DEVICE), b.to(DEVICE)).cpu().numpy()
                           for x, a, b, _ in torch.utils.data.DataLoader(ds, batch_size=256)])

def val_origins(c): return np.arange(H_DATA - c.val_weeks*168, H_DATA - c.pred_len + 1, c.val_stride)

def train(c, seed, fit_end, vo=None, epochs=None, tag=""):
    torch.manual_seed(seed); np.random.seed(seed)
    sc, z, em, dm = prepare(c, fit_end)
    tr = Windows(z, em, dm, np.arange(c.seq_len, fit_end - c.pred_len + 1, c.train_stride), c)
    dl = torch.utils.data.DataLoader(tr, batch_size=c.batch_size, shuffle=True, generator=torch.Generator().manual_seed(seed))
    m = Autoformer(c, em.shape[1], dm.shape[1]).to(DEVICE); opt = torch.optim.Adam(m.parameters(), lr=c.lr)
    va = Windows(z, em, dm, vo, c) if vo is not None else None
    tv = np.stack([y_all[o:o+c.pred_len] for o in vo]) if vo is not None else None
    best, state, bad, log = 1e18, None, 0, []
    for ep in range(1, (epochs or c.max_epochs) + 1):
        m.train(); t0 = time.time()
        for x, a, b, t in dl:
            opt.zero_grad(); F.mse_loss(m(x.to(DEVICE), a.to(DEVICE), b.to(DEVICE)), t.to(DEVICE)).backward()
            if c.grad_clip > 0: nn.utils.clip_grad_norm_(m.parameters(), c.grad_clip)
            opt.step()
        for g in opt.param_groups: g["lr"] *= 0.5
        row = {"epoch": ep, "sec": round(time.time() - t0, 1)}
        if va is not None:
            row.update(metrics(sc.dec(predict(m, va)), tv))
            if row["RMSE"] < best: best, bad, state = row["RMSE"], 0, {k: v.clone() for k, v in m.state_dict().items()}
            else: bad += 1
        log.append(row); print(f"  {tag} epoch {ep}: " + ", ".join(f"{k}={v:.2f}" if isinstance(v, float) else f"{k}={v}" for k, v in row.items() if k != "epoch"), flush=True)
        if va is not None and bad >= c.patience: break
    if state: m.load_state_dict(state)
    return m, sc, log, (z, em, dm)

def evaluate(c, seed):
    vo = val_origins(c)
    m, sc, log, _ = train(c, seed, int(vo[0]), vo, tag=f"{c.name}/seed{seed}")
    b = min(log, key=lambda r: r["RMSE"])
    return {"model": c.name, "covariates": c.covariates, "seed": seed, "RMSE": b["RMSE"], "MAE": b["MAE"],
            "sMAPE": b["sMAPE"], "best_epoch": b["epoch"], "epochs_run": len(log), "params": n_params(m)}

# ---------------- 1) reference baselines on the same validation blocks ----------------
vo0 = val_origins(BASE); T = np.stack([y_all[o:o+168] for o in vo0])
base_rows = [{"model": "baseline: training mean", **metrics(np.full_like(T, y_all[:vo0[0]].mean()), T)},
             {"model": "baseline: persistence", **metrics(np.stack([np.full(168, y_all[o-1]) for o in vo0]), T)},
             {"model": "baseline: last-week mean", **metrics(np.stack([np.full(168, y_all[o-168:o].mean()) for o in vo0]), T)}]
print(pd.DataFrame(base_rows).round(2).to_string(index=False))

# ---------------- 2) search the 5 candidates ----------------
rows = []
for c in CANDIDATES:
    for s in SEEDS:
        rows.append(evaluate(c, s)); print(rows[-1], flush=True)
runs = pd.DataFrame(rows); runs.to_csv(OUT / "candidate_runs.csv", index=False)
summary = (runs.groupby("model").agg(RMSE_mean=("RMSE", "mean"), RMSE_std=("RMSE", "std"), MAE=("MAE", "mean"),
           sMAPE=("sMAPE", "mean"), params=("params", "first"), best_epoch=("best_epoch", "median"))
           .sort_values("RMSE_mean").round(2))
summary.to_csv(OUT / "candidate_summary.csv"); print("\nCANDIDATES (validation, mean over seeds)\n", summary.to_string())

# FIX: select on a pessimistic estimate (mean + std over seeds) so an unstable design cannot win on one lucky seed;
# among designs within 1% of the best, take the fewest parameters.
summary["RMSE_upper"] = (summary["RMSE_mean"] + summary["RMSE_std"].fillna(0)).round(2)
tol = summary["RMSE_upper"].min() * 1.01
winner_name = summary[summary["RMSE_upper"] <= tol].sort_values("params").index[0]
print(summary[["RMSE_mean", "RMSE_std", "RMSE_upper", "params"]].sort_values("RMSE_upper").to_string())
WIN = next(c for c in CANDIDATES if c.name == winner_name)
E_fit = max(1, int(round(summary.loc[winner_name, "best_epoch"])))
print(f"\nWINNER: {winner_name}  (params={summary.loc[winner_name,'params']}, epochs for final fit={E_fit})")

# ---------------- 3) ablation of the external file on the winner ----------------
if RUN_ABLATION:
    ab = []
    for cov in ["none", "past", "past+future"]:
        c = replace(WIN, covariates=cov, name=f"{WIN.name}[{cov}]")
        for s in SEEDS:
            r = evaluate(c, s) if not (cov == WIN.covariates) else runs[(runs.model == WIN.name) & (runs.seed == s)].iloc[0].to_dict()
            r["covariates"] = cov; ab.append(r)
    ab = pd.DataFrame(ab); ab.to_csv(OUT / "ablation_runs.csv", index=False)
    ab_sum = ab.groupby("covariates")[["RMSE", "MAE", "sMAPE"]].agg(["mean", "std"]).round(2)
    ab_sum.to_csv(OUT / "ablation_summary.csv"); print("\nABLATION (external file)\n", ab_sum.to_string())

# ---------------- 4) final model and the 168 forecasts ----------------
win_runs = runs[runs.model == WIN.name]
seed = int(win_runs.sort_values("RMSE").seed.iloc[0])          # FIX: use the winner's best validated seed
print(f"Final seed: {seed}")
if FINAL_MODE == "refit":
    sel_epochs = int(runs[(runs.model == WIN.name) & (runs.seed == seed)].epochs_run.iloc[0])
    m, sc, log, (z, em, dm) = train(WIN, seed, H_DATA, None, epochs=E_fit, tag="FINAL")
    E_declared = sel_epochs + len(log)     # handout: early-stopping run + refit both count
else:
    vo = val_origins(WIN); m, sc, log, (z, em, dm) = train(WIN, seed, int(vo[0]), vo, tag="FINAL")
    E_declared = len(log)
pred = sc.dec(predict(m, Windows(z, em, dm, np.array([H_DATA]), WIN)))[0]
assert pred.shape == (168,) and np.isfinite(pred).all()
P_declared = n_params(m)
pd.DataFrame({"time_idx": np.arange(H_DATA + 1, H_TOTAL + 1), "value": pred}).to_csv(OUT / "submission.csv", index=False)
(OUT / "leaderboard_paste.txt").write_text(", ".join(f"{v:.3f}" for v in pred))
json.dump({"winner": WIN.name, "config": asdict(WIN), "seed": seed, "P_params": P_declared,
           "E_epochs": E_declared, "final_mode": FINAL_MODE}, open(OUT / "submission_info.json", "w"), indent=2)
print(f"\nDONE. Declare P = {P_declared} parameters, E = {E_declared} epochs.")
print(f"Paste file: {OUT/'leaderboard_paste.txt'}  (168 values, time_idx 43657 -> 43824)")
print("First 5 forecasts:", np.round(pred[:5], 1), "...  last:", round(float(pred[-1]), 1))
# FIX: sanity check before pasting - a flat or near-zero week is a red flag
print("Daily means of the forecast:", [round(float(pred[i:i+24].mean()), 1) for i in range(0, 168, 24)])
print("Last observed week mean:", round(float(y_all[-168:].mean()), 1), "| last 10 weeks mean:", round(float(y_all[-1680:].mean()), 1))


                   model   RMSE    MAE  sMAPE
 baseline: training mean 109.42  86.64  92.89
   baseline: persistence 157.13 116.79 101.68
baseline: last-week mean 117.31  92.25  93.37
  A_log_d32/seed0 epoch 1: sec=17.30, RMSE=93.03, MAE=57.65, sMAPE=57.77
  A_log_d32/seed0 epoch 2: sec=16.90, RMSE=88.01, MAE=52.85, sMAPE=53.08
  A_log_d32/seed0 epoch 3: sec=16.80, RMSE=87.12, MAE=51.55, sMAPE=52.80
  A_log_d32/seed0 epoch 4: sec=16.80, RMSE=87.75, MAE=51.73, sMAPE=53.23
  A_log_d32/seed0 epoch 5: sec=16.80, RMSE=87.50, MAE=51.65, sMAPE=53.51
{'model': 'A_log_d32', 'covariates': 'past+future', 'seed': 0, 'RMSE': 87.12106491257349, 'MAE': 51.55404444260611, 'sMAPE': 52.80347870501705, 'best_epoch': 3, 'epochs_run': 5, 'params': 22209}
  A_log_d32/seed1 epoch 1: sec=16.80, RMSE=94.75, MAE=60.52, sMAPE=61.25
  A_log_d32/seed1 epoch 2: sec=16.70, RMSE=86.09, MAE=55.57, sMAPE=56.22
  A_log_d32/seed1 epoch 3: sec=16.70, RMSE=90.10, MAE=55.71, sMAPE=57.20
  A_log_d32/seed1 epoch 4: sec=16.80,

In [3]:
# ===================== TASK 2 - ROUND 3: validated ensemble + full-history refit =====================
# Reuses all definitions from task2_cell.py (no retraining of the old search).
if "train" not in globals():
    exec(open("task2_cell.py").read().split("# ---------------- 1) reference baselines")[0])

RAW_SEEDS = [0, 1, 2, 3, 4]          # raw-target members (best RMSE family)
LOG_SEEDS = [0, 1, 2]                # log-target members (best MAE family)
RAW = replace(BASE, name="raw", target_transform="none")
LOG = replace(BASE, name="log", target_transform="log1p")

vo = val_origins(RAW); fit_end = int(vo[0])
T = np.stack([y_all[o:o+168] for o in vo])
members, val_preds, total_epochs = [], {}, 0

# ---- Phase 1: train every member on the training part, keep its best-epoch validation forecast ----
for cfg, seeds in [(RAW, RAW_SEEDS), (LOG, LOG_SEEDS)]:
    for s in seeds:
        m, sc, log, (z, em, dm) = train(cfg, s, fit_end, vo, tag=f"{cfg.name}/seed{s}")
        best = min(log, key=lambda r: r["RMSE"])
        val_preds[(cfg.name, s)] = sc.dec(predict(m, Windows(z, em, dm, vo, cfg)))
        members.append({"cfg": cfg, "seed": s, "best_epoch": best["epoch"], "val_RMSE": best["RMSE"],
                        "epochs_run": len(log), "params": n_params(m)})
        total_epochs += len(log)

# ---- Choose the ensemble on validation ----
raw_avg = np.mean([val_preds[("raw", s)] for s in RAW_SEEDS], 0)
log_avg = np.mean([val_preds[("log", s)] for s in LOG_SEEDS], 0)
rows = [{"ensemble": "single best raw member", **metrics(val_preds[("raw", min(RAW_SEEDS, key=lambda s: next(m["val_RMSE"] for m in members if m["cfg"].name=="raw" and m["seed"]==s)))], T)},
        {"ensemble": "raw average", **metrics(raw_avg, T)},
        {"ensemble": "log average", **metrics(log_avg, T)}]
best_w, best_r = 1.0, 1e9
for w in np.round(np.arange(0, 1.01, 0.1), 1):              # weight on the raw family
    r = metrics(w * raw_avg + (1 - w) * log_avg, T)
    rows.append({"ensemble": f"blend raw {w:.1f} / log {1-w:.1f}", **r})
    if r["RMSE"] < best_r: best_w, best_r = w, r["RMSE"]
ens = pd.DataFrame(rows).round(2); ens.to_csv(OUT / "ensemble_validation.csv", index=False)
print(ens.to_string(index=False)); print(f"\nCHOSEN raw weight = {best_w}  (validation RMSE {best_r:.2f})")

# ---- Phase 2: refit every member on the FULL history (includes the last 10 weeks) ----
fut_raw, fut_log, P_total = [], [], 0
for mb in members:
    if (mb["cfg"].name == "raw" and best_w == 0) or (mb["cfg"].name == "log" and best_w == 1):
        continue                                            # weight 0 -> member not needed
    m, sc, log, (z, em, dm) = train(mb["cfg"], mb["seed"], H_DATA, None, epochs=mb["best_epoch"],
                                    tag=f"REFIT {mb['cfg'].name}/seed{mb['seed']}")
    f = sc.dec(predict(m, Windows(z, em, dm, np.array([H_DATA]), mb["cfg"])))[0]
    (fut_raw if mb["cfg"].name == "raw" else fut_log).append(f)
    P_total += n_params(m); total_epochs += len(log)

parts = []
if fut_raw: parts.append(best_w * np.mean(fut_raw, 0))
if fut_log: parts.append((1 - best_w) * np.mean(fut_log, 0))
pred = np.clip(np.sum(parts, 0), 0, None)
assert pred.shape == (168,) and np.isfinite(pred).all()

(OUT / "leaderboard_paste_ensemble.txt").write_text(", ".join(f"{v:.3f}" for v in pred))
pd.DataFrame({"time_idx": np.arange(H_DATA + 1, H_TOTAL + 1), "value": pred}).to_csv(OUT / "submission_ensemble.csv", index=False)
json.dump({"raw_weight": best_w, "raw_seeds": RAW_SEEDS, "log_seeds": LOG_SEEDS, "P_params": P_total,
           "E_epochs": total_epochs, "val_RMSE": best_r}, open(OUT / "submission_ensemble_info.json", "w"), indent=2)
print(f"\nDeclare P = {P_total}, E = {total_epochs}  (all members, selection + refit epochs)")
print("Daily means:", [round(float(pred[i:i+24].mean()), 1) for i in range(0, 168, 24)])
print("Paste file:", OUT / "leaderboard_paste_ensemble.txt")

  raw/seed0 epoch 1: sec=17.10, RMSE=76.62, MAE=56.52, sMAPE=74.44
  raw/seed0 epoch 2: sec=16.90, RMSE=75.60, MAE=57.78, sMAPE=75.47
  raw/seed0 epoch 3: sec=16.80, RMSE=74.72, MAE=57.58, sMAPE=76.89
  raw/seed0 epoch 4: sec=16.80, RMSE=79.98, MAE=63.97, sMAPE=78.18
  raw/seed0 epoch 5: sec=16.80, RMSE=80.13, MAE=63.63, sMAPE=78.17
  raw/seed1 epoch 1: sec=17.00, RMSE=87.82, MAE=70.70, sMAPE=83.91
  raw/seed1 epoch 2: sec=17.50, RMSE=92.23, MAE=76.75, sMAPE=84.30
  raw/seed1 epoch 3: sec=16.90, RMSE=96.50, MAE=81.12, sMAPE=86.43
  raw/seed2 epoch 1: sec=16.90, RMSE=81.29, MAE=59.69, sMAPE=81.40
  raw/seed2 epoch 2: sec=16.70, RMSE=77.86, MAE=59.76, sMAPE=76.51
  raw/seed2 epoch 3: sec=16.70, RMSE=73.82, MAE=55.34, sMAPE=76.50
  raw/seed2 epoch 4: sec=16.80, RMSE=74.71, MAE=58.15, sMAPE=76.06
  raw/seed2 epoch 5: sec=16.80, RMSE=73.77, MAE=56.16, sMAPE=75.81
  raw/seed2 epoch 6: sec=16.80, RMSE=73.73, MAE=56.65, sMAPE=75.64
  raw/seed2 epoch 7: sec=16.80, RMSE=73.72, MAE=56.60, sMAPE=7